In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2


# วิธี A: แนวทางคลาสิก (The Classical Approach)

In [2]:
# โหลด dataset
df = pd.read_csv("NLP Dataset/complete_dataset_(Labeling).csv")

texts = df["Sentence"]
labels = df["Sentiment"]

# แบ่ง train / test
x_train_A, x_test_A, y_train_A, y_test_A = train_test_split(
    texts, labels, test_size=0.2, random_state=12, stratify=labels
)

# Vectorize
vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
x_train = vectorizer.fit_transform(x_train_A)

# Train a model
lr_model = LogisticRegression(class_weight="balanced",max_iter=1000)
lr_model.fit(x_train, y_train_A)
print("Model trained successfully!")


Model trained successfully!


# วิธี B: แนวทาง Deep Learning (The Neural Approach)

In [3]:
# โหลด dataset
df = pd.read_csv("NLP Dataset/complete_dataset_(Labeling).csv")

texts = df["Sentence"]

df["Sentiment"] = df["Sentiment"].map({
    "negative": 0,
    "neutral": 1,
    "positive": 2
})

labels = df["Sentiment"]

# แบ่ง train / test
x_train_B, x_test_B, y_train_B, y_test_B = train_test_split(
    texts, labels, test_size=0.2, random_state=21, stratify=labels
)

# Tokenization
max_words = 5000
max_len = 100

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(x_train_B)

x_train_seqB = tokenizer.texts_to_sequences(x_train_B)
x_test_seqB = tokenizer.texts_to_sequences(x_test_B)

x_train_padB = pad_sequences(x_train_seqB, maxlen=max_len, padding="post")
x_test_padB = pad_sequences(x_test_seqB, maxlen=max_len, padding="post")

# Validation split
x_train_subB, x_val, y_train_subB, y_val = train_test_split(
    x_train_padB,
    y_train_B,
    test_size=0.2,
    random_state=21,
    stratify=y_train_B
)

model = Sequential([
    Embedding(max_words, 128, input_length=max_len),

    Bidirectional(LSTM(32, dropout=0.3, recurrent_dropout=0.3)),

    Dense(64, activation="relu", kernel_regularizer=l2(0.001)),
    Dropout(0.5),

    Dense(3, activation="softmax")
])

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=Adam(learning_rate=0.0003),
    metrics=["accuracy"]
)

# Early stopping
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

# Class weight
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train_B),
    y=y_train_B
)

class_weights = dict(enumerate(class_weights))

# Train a model
history = model.fit(
    x_train_subB,
    y_train_subB,
    epochs=5,
    batch_size=32,
    validation_data=(x_val, y_val),
    callbacks=[early_stop],
    class_weight=class_weights
)

print("Model trained successfully!")


Epoch 1/5
134/134 [==============================] - 59s 378ms/step - loss: 1.1474 - accuracy: 0.4599 - val_loss: 1.1229 - val_accuracy: 0.5482
Epoch 2/5
134/134 [==============================] - 58s 427ms/step - loss: 1.0654 - accuracy: 0.5600 - val_loss: 0.9193 - val_accuracy: 0.6520
Epoch 3/5
134/134 [==============================] - 57s 430ms/step - loss: 0.9087 - accuracy: 0.6650 - val_loss: 0.8983 - val_accuracy: 0.6604
Epoch 4/5
134/134 [==============================] - 60s 444ms/step - loss: 0.7301 - accuracy: 0.7490 - val_loss: 0.8464 - val_accuracy: 0.6623
Epoch 5/5
134/134 [==============================] - 60s 448ms/step - loss: 0.5548 - accuracy: 0.8178 - val_loss: 0.8713 - val_accuracy: 0.6642
Model trained successfully!


In [4]:
# Evaluate A
x_test_A = vectorizer.transform(x_test_A)
predictions = lr_model.predict(x_test_A)

print(classification_report(y_test_A, predictions))

              precision    recall  f1-score   support

    negative       0.39      0.47      0.42       180
     neutral       0.77      0.73      0.75       728
    positive       0.70      0.71      0.70       429

    accuracy                           0.69      1337
   macro avg       0.62      0.63      0.63      1337
weighted avg       0.70      0.69      0.69      1337



In [5]:
# Evaluate B
lstm_probs = model.predict(x_test_padB)
lstm_pred = np.argmax(lstm_probs, axis=1)

print(classification_report(y_test_B, lstm_pred))

42/42 [==============================] - 5s 112ms/step
              precision    recall  f1-score   support

           0       0.30      0.52      0.38       180
           1       0.81      0.69      0.74       728
           2       0.71      0.69      0.70       429

    accuracy                           0.66      1337
   macro avg       0.61      0.63      0.61      1337
weighted avg       0.71      0.66      0.68      1337

